# Strategy 1 - Optimal K Selection and Fair Clustering Evaluation

This notebook implements two approaches for determining optimal k values and evaluates clustering methods:
- Standard K-Means
- Fairlet Decomposition
- Binary Fair K-Means (BFKM)
- FAIR-CENTROID CLUSTERING（CCF）
- FAIRNESS-AWARE POSTPROCESSING （PP-NFP/PP-Gini）
- RAWLSIAN K-Means

Datasets: Adult, COMPAS, German Credit, Default Credit Card, Law School

In [2]:
import numpy as np
import pandas as pd
import pickle
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    silhouette_score,
    silhouette_samples,
    calinski_harabasz_score,
    davies_bouldin_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
    adjusted_mutual_info_score,
    fowlkes_mallows_score,
    v_measure_score
)
from scipy.spatial.distance import cdist
from scipy.stats import entropy
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict, Counter
import warnings

warnings.filterwarnings('ignore')
# Set random seed for reproducibility
np.random.seed(0)

## 1. Data Loading and Preprocessing

In [4]:
# DATA LOADING
def load_adult():
    print("\n" + "="*80)
    print("Loading Adult Income Dataset")
    print("="*80)
    
    columns = [
        'age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status',
        'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss',
        'hours-per-week', 'native-country', 'income'
    ]
    
    adult = pd.read_csv("./Dataset/adult/adult.data", names=columns, header=None)
    print(f"Shape: {adult.shape}")
    
    #remove rows with missing values
    adult = adult.replace(' ?', np.nan)
    adult = adult.dropna()
    
    # Sensitive attribute - gender
    adult['sex_binary'] = (adult['sex'].str.strip() == 'Male').astype(int)
    
    # Features for clustering (exclude sensitive and target)
    adult_features = [c for c in adult.columns if c not in ["income", "sex", "race", "sex_binary"]]
    
    print(f"Sex distribution: {adult['sex'].value_counts().to_dict()}")
    print(f"Binary encoding - 0 (Female): {(adult['sex_binary']==0).sum()}, 1 (Male): {(adult['sex_binary']==1).sum()}")
    
    return adult, adult_features, 'sex_binary'


def load_compas():
    print("\n" + "="*80)
    print("Loading COMPAS Dataset")
    print("="*80)
    
    compas = pd.read_csv("./Dataset/compas/compas-scores.csv")
    print(f"Shape: {compas.shape}")
    
    # Filter data
    compas = compas[
        (compas['days_b_screening_arrest'] <= 30) &
        (compas['days_b_screening_arrest'] >= -30) &
        (compas['is_recid'] != -1)
    ]
    
    # Remove missing values columns
    selected_cols = compas.columns[compas.isnull().sum() == 0].tolist()
    compas = compas[selected_cols]
    
    print(f"Shape after filtering: {compas.shape}")
    
    # Sensitive attribute - gender
    compas['sex_binary'] = (compas['sex'] == 'Male').astype(int)
    
    compas_features = [c for c in compas.columns if c not in ["is_recid", "sex", "race", "sex_binary"]]
    print(f"Race distribution:\n{compas['sex'].value_counts()}")
    print(f"Binary encoding - 0 (Female): {(compas['sex_binary']==0).sum()}, 1 (Male): {(compas['sex_binary']==1).sum()}")
    
    return compas, compas_features, 'sex_binary'


def load_german():
    print("\n" + "="*80)
    print("Loading German Credit Dataset")
    print("="*80)
    
    german_columns = [
        'checking_status', 'duration', 'credit_history', 'purpose', 'credit_amount',
        'savings_status', 'employment', 'installment_rate', 'personal_status_sex',
        'other_parties', 'residence_since', 'property_magnitude', 'age',
        'other_payment_plans', 'housing', 'existing_credits', 'job',
        'num_dependents', 'own_telephone', 'foreign_worker', 'class'
    ]
    
    german = pd.read_csv("./Dataset/german/german.data", 
                         sep=' ', 
                         names=german_columns, 
                         header=None)
    
    print(f"Shape: {german.shape}")
    
    # Extract gender from personal_status_sex
    def extract_sex(status):
        if status in ['A91', 'A93', 'A94']:
            return 'male'
        elif status in ['A92', 'A95']:
            return 'female'
        return 'unknown'
    
    german['sex'] = german['personal_status_sex'].apply(extract_sex)
    german = german[german['sex'] != 'unknown']  # Remove unknown
    german['sex_binary'] = (german['sex'] == 'male').astype(int)
    
    german_features = [c for c in german.columns if c not in ["class", "sex", "sex_binary"]]
    
    print(f"Sex distribution: {german['sex'].value_counts().to_dict()}")
    print(f"Binary encoding - 0 (Female): {(german['sex_binary']==0).sum()}, 1 (Male): {(german['sex_binary']==1).sum()}")
    
    return german, german_features, 'sex_binary'


def load_credit():
    print("\n" + "="*80)
    print("Loading Default Credit Card Dataset")
    print("="*80)
    
    credit = pd.read_csv("./Dataset/default of credit card clients/default of credit card clients.csv",
                         header=0)
    
    # Rename columns
    if 'X1' in credit.columns or credit.columns[0] == 'ID':
        credit.columns = [
            'ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE',
            'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6',
            'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6',
            'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6',
            'default'
        ]
    
    credit = credit.drop(columns=['ID'])
    print(f"Shape: {credit.shape}")
    
    # Extract gender
    credit['SEX'] = pd.to_numeric(credit['SEX'], errors='coerce')
    credit['sex'] = credit['SEX'].map({1.0: 'male', 2.0: 'female'})
    credit = credit.dropna(subset=['sex'])
    
    credit['sex_binary'] = (credit['sex'] == 'male').astype(int)
    credit_features = [c for c in credit.columns if c not in ["default", "sex", "SEX", "sex_binary"]]
    
    print(f"Sex distribution: {credit['sex'].value_counts().to_dict()}")
    print(f"Binary encoding - 0 (Female): {(credit['sex_binary']==0).sum()}, 1 (Male): {(credit['sex_binary']==1).sum()}")
    
    return credit, credit_features, 'sex_binary'


def load_law():
    print("\n" + "="*80)
    print("Loading Law School (LSAC) Dataset")
    print("="*80)
    
    lsac = pd.read_csv("./Dataset/law/law_dataset.csv")
    print(f"LSAC shape: {lsac.shape}")
    print(f"Columns: {lsac.columns.tolist()}")
    
    # Check male column
    print(f"\nMale column dtype: {lsac['male'].dtype}")
    print(f"Male values: {lsac['male'].unique()}")
    print(f"Male distribution:\n{lsac['male'].value_counts()}")
    
    # Convert male column to standard gender column
    lsac['sex'] = lsac['male'].map({
        1: 'Male',
        0: 'Female',
        1.0: 'Male',
        0.0: 'Female'
    })
    
    print(f"\nStandardized sex distribution:")
    print(lsac['sex'].value_counts())
    
    print(f"\nMissing values:")
    missing = lsac.isnull().sum()
    if missing.sum() > 0:
        print(missing[missing > 0].sort_values(ascending=False))
    else:
        print("✓ No missing values!")
    
    if lsac['sex'].isnull().any():
        print(f"\n Warning: {lsac['sex'].isnull().sum()} rows have null sex values")
        print("These will be dropped")
        lsac = lsac[lsac['sex'].notna()]
    
    lsac_clean = lsac.copy()
    print(f"\nCleaned data shape: {lsac_clean.shape}")
    

    lsac_clean['sex_binary'] = (lsac_clean['sex'] == 'Male').astype(int)
    
    # Define features
    drop_cols = ['male', 'sex', 'pass_bar', 'sex_binary']
    lsac_features = [c for c in lsac_clean.columns if c not in drop_cols]
    
    print(f"Binary encoding - 0 (Female): {(lsac_clean['sex_binary']==0).sum()}, 1 (Male): {(lsac_clean['sex_binary']==1).sum()}")
    return lsac_clean, lsac_features, 'sex_binary'

In [5]:
def preprocess_dataset(df, feature_cols, sensitive_col):
    # Extract features
    X_df = df[feature_cols].copy()
    
    # Encode categorical variables
    categorical_cols = X_df.select_dtypes(include=['object']).columns.tolist()
    
    if categorical_cols:
        print(f"  Encoding {len(categorical_cols)} categorical columns...")
        le = LabelEncoder()
        for col in categorical_cols:
            X_df[col] = le.fit_transform(X_df[col].astype(str))
    
    # Convert to numpy array
    X = X_df.values.astype(float)
    
    # Check for non-numeric values
    if np.any(np.isnan(X)) or np.any(np.isinf(X)):
        print("  Warning: Found NaN or Inf values, filling with 0")
        X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    
    # Standardize features
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    
    # Extract sensitive attribute
    sensitive_attr = df[sensitive_col].values.astype(int)
    
    print(f"  Final shape: {X.shape}")
    print(f"  Sensitive attribute distribution: 0={np.sum(sensitive_attr==0)}, 1={np.sum(sensitive_attr==1)}")
    
    return X, sensitive_attr

In [6]:
# Load all datasets
print("\n" + "="*80)
print("LOADING ALL DATASETS")
print("="*80)

datasets = {}
dataset_loaders = [
    ('adult', load_adult),
    ('compas', load_compas),
    ('german', load_german),
    ('credit', load_credit),
    ('law', load_law)
]

for name, loader in dataset_loaders:
    try:
        df, features, sensitive_col = loader()
        print(f"\nProcessing {name.upper()}...")
        X, sensitive = preprocess_dataset(df, features, sensitive_col)
        datasets[name] = {
            'X': X,
            'sensitive': sensitive,
            'n_samples': X.shape[0],
            'n_features': X.shape[1]
        }
    except Exception as e:
        print(f" Error loading {name}: {e}")

print("\n" + "="*80)
print("DATASET SUMMARY")
print("="*80)
for name, data in datasets.items():
    print(f"{name.upper()}: {data['n_samples']} samples, {data['n_features']} features")

dataset_names = list(datasets.keys())
print(f"\n Total datasets loaded: {len(dataset_names)}")


LOADING ALL DATASETS

Loading Adult Income Dataset
Shape: (32561, 15)
Sex distribution: {' Male': 20380, ' Female': 9782}
Binary encoding - 0 (Female): 9782, 1 (Male): 20380

Processing ADULT...
  Encoding 6 categorical columns...
  Final shape: (30162, 12)
  Sensitive attribute distribution: 0=9782, 1=20380

Loading COMPAS Dataset
Shape: (11757, 47)
Shape after filtering: (9395, 30)
Race distribution:
sex
Male      7462
Female    1933
Name: count, dtype: int64
Binary encoding - 0 (Female): 1933, 1 (Male): 7462

Processing COMPAS...
  Encoding 15 categorical columns...
  Final shape: (9395, 27)
  Sensitive attribute distribution: 0=1933, 1=7462

Loading German Credit Dataset
Shape: (1000, 21)
Sex distribution: {'male': 690, 'female': 310}
Binary encoding - 0 (Female): 310, 1 (Male): 690

Processing GERMAN...
  Encoding 13 categorical columns...
  Final shape: (1000, 20)
  Sensitive attribute distribution: 0=310, 1=690

Loading Default Credit Card Dataset
Shape: (30001, 24)
Sex distrib

## 2. Optimal K Selection - Combined Score

In [8]:
def normalize_metric(values, inverse=False):
    values = np.array(values)
    min_val, max_val = np.min(values), np.max(values)
    
    if max_val - min_val == 0:
        return np.ones_like(values) * 0.5
    
    normalized = (values - min_val) / (max_val - min_val)
    
    if inverse:
        normalized = 1 - normalized
    
    return normalized


def approach1_optimal_k(X, k_range=(2, 11)):
    results = {
        'k_values': [],
        'sse': [],
        'silhouette': [],
        'calinski': [],
        'dbi': [],
        'silhouette_std': []
    }
    
    for k in range(k_range[0], k_range[1]):
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = kmeans.fit_predict(X)
        
        # SSE
        sse = kmeans.inertia_
        
        # Silhouette Score
        sil_score = silhouette_score(X, labels)
        sil_samples = silhouette_samples(X, labels)
        sil_std = np.std(sil_samples)
        
        # Calinski-Harabasz Index
        ch_score = calinski_harabasz_score(X, labels)
        
        # Davies-Bouldin Index
        db_score = davies_bouldin_score(X, labels)
        
        results['k_values'].append(k)
        results['sse'].append(sse)
        results['silhouette'].append(sil_score)
        results['calinski'].append(ch_score)
        results['dbi'].append(db_score)
        results['silhouette_std'].append(sil_std)
    
    # Normalize metrics
    norm_sse = normalize_metric(results['sse'], inverse=True)
    norm_sil = normalize_metric(results['silhouette'], inverse=False)
    norm_ch = normalize_metric(results['calinski'], inverse=False)
    norm_dbi = normalize_metric(results['dbi'], inverse=True)
    norm_sil_std = normalize_metric(results['silhouette_std'], inverse=True)
    
    # Combined score (average of normalized metrics)
    combined_scores = (norm_sse + norm_sil + norm_ch + norm_dbi + norm_sil_std) / 5
    
    # Find optimal k
    optimal_idx = np.argmax(combined_scores)
    optimal_k = results['k_values'][optimal_idx]
    
    results['combined_scores'] = combined_scores
    results['optimal_k'] = optimal_k
    
    return results

## 3. Optimal K Selection - Multiple Methods

In [10]:
# WCSS
def elbow_method(X, k_range=(2, 11)):
    wcss = []
    for k in range(k_range[0], k_range[1]):
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(X)
        wcss.append(kmeans.inertia_)
    
    # Find elbow using second derivative
    wcss = np.array(wcss)
    diff1 = np.diff(wcss)
    diff2 = np.diff(diff1)
    elbow_idx = np.argmax(diff2) + 2
    optimal_k = range(k_range[0], k_range[1])[elbow_idx] if elbow_idx < len(range(k_range[0], k_range[1])) else k_range[0]
    
    return optimal_k, wcss

# Silhouette
def silhouette_method(X, k_range=(2, 11)):
    silhouette_scores = []
    for k in range(k_range[0], k_range[1]):
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = kmeans.fit_predict(X)
        score = silhouette_score(X, labels)
        silhouette_scores.append(score)
    
    optimal_idx = np.argmax(silhouette_scores)
    optimal_k = range(k_range[0], k_range[1])[optimal_idx]
    
    return optimal_k, silhouette_scores

# Calinski-Harabasz
def calinski_harabasz_method(X, k_range=(2, 11)):
    ch_scores = []
    for k in range(k_range[0], k_range[1]):
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = kmeans.fit_predict(X)
        score = calinski_harabasz_score(X, labels)
        ch_scores.append(score)
    
    optimal_idx = np.argmax(ch_scores)
    optimal_k = range(k_range[0], k_range[1])[optimal_idx]
    
    return optimal_k, ch_scores

# Davies-Bouldin
def davies_bouldin_method(X, k_range=(2, 11)):
    db_scores = []
    for k in range(k_range[0], k_range[1]):
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = kmeans.fit_predict(X)
        score = davies_bouldin_score(X, labels)
        db_scores.append(score)
    
    optimal_idx = np.argmin(db_scores)  # Lower is better
    optimal_k = range(k_range[0], k_range[1])[optimal_idx]
    
    return optimal_k, db_scores

# Gap Statistic method
def gap_statistic(X, k_range=(2, 11), n_refs=10):
    gaps = []
    
    for k in range(k_range[0], k_range[1]):
        # Actual clustering
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = kmeans.fit_predict(X)
        wk = kmeans.inertia_
        
        # Reference distributions
        ref_disps = []
        for _ in range(n_refs):
            # Generate random reference data
            random_data = np.random.uniform(X.min(axis=0), X.max(axis=0), size=X.shape)
            ref_kmeans = KMeans(n_clusters=k, random_state=None, n_init=10)
            ref_kmeans.fit(random_data)
            ref_disps.append(ref_kmeans.inertia_)
        
        gap = np.log(np.mean(ref_disps)) - np.log(wk)
        gaps.append(gap)
    
    optimal_idx = np.argmax(gaps)
    optimal_k = range(k_range[0], k_range[1])[optimal_idx]
    
    return optimal_k, gaps

# Stability analysis using ARI
def stability_analysis(X, k_range=(2, 11), n_iterations=10, sample_ratio=0.8):
    stability_scores = []
    n_samples = int(len(X) * sample_ratio)
    
    for k in range(k_range[0], k_range[1]):
        ari_scores = []
        
        for _ in range(n_iterations):
            # Two random samples
            idx1 = np.random.choice(len(X), n_samples, replace=False)
            idx2 = np.random.choice(len(X), n_samples, replace=False)
            
            # Common indices
            common_idx = np.intersect1d(idx1, idx2)
            
            if len(common_idx) > k:
                kmeans1 = KMeans(n_clusters=k, random_state=None, n_init=10)
                kmeans2 = KMeans(n_clusters=k, random_state=None, n_init=10)
                
                labels1 = kmeans1.fit_predict(X[idx1])
                labels2 = kmeans2.fit_predict(X[idx2])
                
                # Map common indices
                common_idx1 = np.array([np.where(idx1 == i)[0][0] for i in common_idx])
                common_idx2 = np.array([np.where(idx2 == i)[0][0] for i in common_idx])
                
                ari = adjusted_rand_score(labels1[common_idx1], labels2[common_idx2])
                ari_scores.append(ari)
        
        stability = np.mean(ari_scores) if ari_scores else 0
        stability_scores.append(stability)
    
    optimal_idx = np.argmax(stability_scores)
    optimal_k = range(k_range[0], k_range[1])[optimal_idx]
    
    return optimal_k, stability_scores


def approach2_optimal_k(X, k_range=(2, 11)):
    results = {}
    
    print("  Running Elbow Method...")
    results['elbow_k'], results['elbow_scores'] = elbow_method(X, k_range)
    
    print("  Running Silhouette Method...")
    results['silhouette_k'], results['silhouette_scores'] = silhouette_method(X, k_range)
    
    print("  Running Calinski-Harabasz Method...")
    results['calinski_k'], results['calinski_scores'] = calinski_harabasz_method(X, k_range)
    
    print("  Running Davies-Bouldin Method...")
    results['davies_k'], results['davies_scores'] = davies_bouldin_method(X, k_range)
    
    print("  Running Gap Statistic...")
    results['gap_k'], results['gap_scores'] = gap_statistic(X, k_range)
    
    print("  Running Stability Analysis...")
    results['stability_k'], results['stability_scores'] = stability_analysis(X, k_range)
    
    k_values = [
        results['elbow_k'],
        results['silhouette_k'],
        results['calinski_k'],
        results['davies_k'],
        results['gap_k'],
        results['stability_k']
    ]
    
    k_counts = Counter(k_values)
    results['majority_vote_k'] = k_counts.most_common(1)[0][0]
    results['k_distribution'] = dict(k_counts)
    
    return results

## 4. Summary Table of Optimal K Values

In [12]:
print("\n" + "="*80)
print("APPROACH 1: COMBINED SCORE METHOD")
print("="*80)

approach1_results = {}

for name, data in datasets.items():
    print(f"\n{name.upper()}:")
    results = approach1_optimal_k(data['X'])
    approach1_results[name] = results
    print(f"  Optimal k = {results['optimal_k']}")
    print(f"  Silhouette at optimal k = {results['silhouette'][results['optimal_k']-2]:.4f}")


print("\n" + "="*80)
print("APPROACH 2: MULTIPLE METHODS")
print("="*80)

approach2_results = {}

for name, data in datasets.items():
    print(f"\n{name.upper()}:")
    results = approach2_optimal_k(data['X'])
    approach2_results[name] = results
    print(f"\n  Optimal k values:")
    print(f"    Elbow: {results['elbow_k']}")
    print(f"    Silhouette: {results['silhouette_k']}")
    print(f"    Calinski-Harabasz: {results['calinski_k']}")
    print(f"    Davies-Bouldin: {results['davies_k']}")
    print(f"    Gap Statistic: {results['gap_k']}")
    print(f"    Stability: {results['stability_k']}")
    print(f"    Majority Vote: {results['majority_vote_k']}")


# Create summary table
summary_data = []

for name in dataset_names:
    if name in approach1_results and name in approach2_results:
        row = {
            'Dataset': name.upper(),
            'Approach 1 (Combined)': approach1_results[name]['optimal_k'],
            'Elbow': approach2_results[name]['elbow_k'],
            'Silhouette': approach2_results[name]['silhouette_k'],
            'Calinski-Harabasz': approach2_results[name]['calinski_k'],
            'Davies-Bouldin': approach2_results[name]['davies_k'],
            'Gap Statistic': approach2_results[name]['gap_k'],
            'Stability': approach2_results[name]['stability_k'],
            'Approach 2 (Majority)': approach2_results[name]['majority_vote_k']
        }
        summary_data.append(row)

summary_df = pd.DataFrame(summary_data)
print("\n" + "="*120)
print("OPTIMAL K VALUES SUMMARY")
print("="*120)
print(summary_df.to_string(index=False))

summary_df.to_csv('optimal_k_summary.csv', index=False)
print("\n✓ Summary saved to: optimal_k_summary.csv")


APPROACH 1: COMBINED SCORE METHOD

ADULT:
  Optimal k = 5
  Silhouette at optimal k = 0.1584

COMPAS:
  Optimal k = 2
  Silhouette at optimal k = 0.1794

GERMAN:
  Optimal k = 6
  Silhouette at optimal k = 0.0913

CREDIT:
  Optimal k = 2
  Silhouette at optimal k = 0.1383

LAW:
  Optimal k = 4
  Silhouette at optimal k = 0.2817

APPROACH 2: MULTIPLE METHODS

ADULT:
  Running Elbow Method...
  Running Silhouette Method...
  Running Calinski-Harabasz Method...
  Running Davies-Bouldin Method...
  Running Gap Statistic...
  Running Stability Analysis...

  Optimal k values:
    Elbow: 6
    Silhouette: 5
    Calinski-Harabasz: 5
    Davies-Bouldin: 9
    Gap Statistic: 10
    Stability: 2
    Majority Vote: 5

COMPAS:
  Running Elbow Method...
  Running Silhouette Method...
  Running Calinski-Harabasz Method...
  Running Davies-Bouldin Method...
  Running Gap Statistic...
  Running Stability Analysis...

  Optimal k values:
    Elbow: 4
    Silhouette: 2
    Calinski-Harabasz: 2
    Davi

## 5. Fair Clustering Algorithms Implementation

In [14]:
# 1. STANDARD K-MEANS
def standard_kmeans(X, k, random_state=42):
    kmeans = KMeans(n_clusters=k, random_state=random_state, n_init=10)
    labels = kmeans.fit_predict(X)
    centers = kmeans.cluster_centers_
    return labels, centers


# 2. FAIRLET DECOMPOSITION
def create_fairlets(X, sensitive_attr, p, q):
    n = len(X)
    group0_idx = np.where(sensitive_attr == 0)[0]
    group1_idx = np.where(sensitive_attr == 1)[0]
    
    fairlets = []
    fairlet_labels = np.full(n, -1)
    fairlet_id = 0
    
    used_0 = set()
    used_1 = set()
    
    # Create balanced fairlets
    while len(used_0) + p <= len(group0_idx) and len(used_1) + q <= len(group1_idx):
        # Select p points from group 0
        available_0 = [i for i in group0_idx if i not in used_0]
        selected_0 = np.random.choice(available_0, p, replace=False)
        
        # Select q points from group 1
        available_1 = [i for i in group1_idx if i not in used_1]
        selected_1 = np.random.choice(available_1, q, replace=False)
        
        # Create fairlet
        fairlet_members = np.concatenate([selected_0, selected_1])
        fairlets.append(fairlet_members)
        fairlet_labels[fairlet_members] = fairlet_id
        
        used_0.update(selected_0)
        used_1.update(selected_1)
        fairlet_id += 1
    
    # Handle remaining points
    remaining = np.where(fairlet_labels == -1)[0]
    if len(remaining) > 0:
        fairlets.append(remaining)
        fairlet_labels[remaining] = fairlet_id
    
    return fairlets, fairlet_labels


def fairlet_clustering(X, sensitive_attr, k, p=1, q=1):
    fairlets, fairlet_labels = create_fairlets(X, sensitive_attr, p, q)
    
    fairlet_centers = []
    for i, fairlet in enumerate(fairlets):
        center = np.mean(X[fairlet], axis=0)
        fairlet_centers.append(center)
    fairlet_centers = np.array(fairlet_centers)
    
    # Cluster fairlet centers
    if len(fairlet_centers) >= k:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        fairlet_cluster_labels = kmeans.fit_predict(fairlet_centers)
    else:
        fairlet_cluster_labels = np.arange(len(fairlet_centers))
    
    # Map points to clusters
    labels = np.zeros(len(X), dtype=int)
    for i, cluster_label in enumerate(fairlet_cluster_labels):
        labels[fairlets[i]] = cluster_label
    
    # Compute final cluster centers
    centers = []
    for i in range(k):
        cluster_points = X[labels == i]
        if len(cluster_points) > 0:
            centers.append(np.mean(cluster_points, axis=0))
        else:
            centers.append(np.zeros(X.shape[1]))
            
    return labels, np.array(centers)


# 3. BINARY FAIR K-MEANS (BFKM)
def bfkm_clustering(X, sensitive_attr, k, max_iter=100, balance_weight=1.0):
    n_samples = len(X)
    
    # Initialize centers randomly
    center_indices = np.random.choice(n_samples, k, replace=False)
    centers = X[center_indices].copy()
    
    for iteration in range(max_iter):
        old_centers = centers.copy()
        labels = np.zeros(n_samples, dtype=int)
        distances = cdist(X, centers, metric='euclidean')
        
        for i in range(k):
            labels[np.argmin(distances, axis=1) == i] = i
        
        for cluster_id in range(k):
            cluster_mask = (labels == cluster_id)
            cluster_sensitive = sensitive_attr[cluster_mask]
            
            if len(cluster_sensitive) == 0:
                continue
            
            n_group0 = np.sum(cluster_sensitive == 0)
            n_group1 = np.sum(cluster_sensitive == 1)
            
            # Calculate imbalance
            total_group0 = np.sum(sensitive_attr == 0)
            total_group1 = np.sum(sensitive_attr == 1)
            
            if total_group0 > 0 and total_group1 > 0:
                expected_ratio = total_group1 / total_group0
                actual_ratio = n_group1 / max(n_group0, 1)
                
                if abs(actual_ratio - expected_ratio) > 0.3:
                    cluster_indices = np.where(cluster_mask)[0]
                    
                    if actual_ratio > expected_ratio:
                        group1_in_cluster = cluster_indices[sensitive_attr[cluster_indices] == 1]
                        if len(group1_in_cluster) > 0:
                            dists = distances[group1_in_cluster, cluster_id]
                            farthest_idx = group1_in_cluster[np.argmax(dists)]
                            sorted_clusters = np.argsort(distances[farthest_idx])
                            labels[farthest_idx] = sorted_clusters[1] if sorted_clusters[1] != cluster_id else sorted_clusters[2]
                    else:
                        group0_in_cluster = cluster_indices[sensitive_attr[cluster_indices] == 0]
                        if len(group0_in_cluster) > 0:
                            dists = distances[group0_in_cluster, cluster_id]
                            farthest_idx = group0_in_cluster[np.argmax(dists)]
                            sorted_clusters = np.argsort(distances[farthest_idx])
                            labels[farthest_idx] = sorted_clusters[1] if sorted_clusters[1] != cluster_id else sorted_clusters[2]
        
        # Update centers
        for i in range(k):
            cluster_points = X[labels == i]
            if len(cluster_points) > 0:
                centers[i] = np.mean(cluster_points, axis=0)
        
        # Check convergence
        if np.allclose(centers, old_centers, rtol=1e-4):
            break
    
    return labels, centers


# 4. FAIR-CENTROID CLUSTERING
class OptimizedFairCentroid:
    def __init__(self, n_clusters=6, max_iter=50, min_iter=10, 
                 tol=0.005, random_state=0, candidate_fraction=0.05):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.min_iter = min_iter
        self.tol = tol
        self.random_state = random_state
        self.candidate_fraction = candidate_fraction
        
        self.labels_ = None
        self.cluster_centers_ = None
        self.inertia_ = None
        self.n_iter_ = 0
    
    def _compute_cluster_fairness_fast(self, X_cluster, S_cluster, centroid):
        if len(X_cluster) == 0:
            return 0.0
        
        unique_groups = np.unique(S_cluster)
        if len(unique_groups) < 2:
            return 0.0
        
        # Distance
        distances_sq = np.sum((X_cluster - centroid) ** 2, axis=1)
        
        # Group means
        group_means = []
        for g in unique_groups:
            mask_g = (S_cluster == g)
            if mask_g.sum() > 0:
                group_means.append(distances_sq[mask_g].mean())
        
        if len(group_means) < 2:
            return 0.0
        
        # Disparity = max - min
        return max(group_means) - min(group_means)
    
    def _precompute_cluster_stats(self, X, labels, S, centers):
        cluster_stats = {}
        
        for c in range(self.n_clusters):
            mask_c = (labels == c)
            n_c = mask_c.sum()
            
            if n_c == 0:
                cluster_stats[c] = {
                    'n': 0,
                    'centroid': centers[c],
                    'fairness': 0.0,
                    'X': np.array([]),
                    'S': np.array([])
                }
                continue
            
            X_c = X[mask_c]
            S_c = S[mask_c]
            centroid_c = centers[c]
            
            fairness_c = self._compute_cluster_fairness_fast(X_c, S_c, centroid_c)
            
            cluster_stats[c] = {
                'n': n_c,
                'centroid': centroid_c,
                'fairness': fairness_c,
                'X': X_c,
                'S': S_c
            }
        
        return cluster_stats
    
    def _find_candidates(self, X, labels, centers, n_candidates):
        n = len(X)
        
        # Compute distance matrix
        distances = np.zeros((n, self.n_clusters))
        for c in range(self.n_clusters):
            distances[:, c] = np.linalg.norm(X - centers[c], axis=1)
        
        # Distance to current cluster
        current_dists = distances[np.arange(n), labels]
        
        # Distance to nearest alternative cluster
        alt_distances = distances.copy()
        alt_distances[np.arange(n), labels] = np.inf
        nearest_alt_dists = np.min(alt_distances, axis=1)
        
        # how much closer to new cluster
        benefit = current_dists - nearest_alt_dists
        
        # Select top candidates
        candidate_indices = np.argsort(-benefit)[:n_candidates]
        
        return candidate_indices
    
    def fit(self, X, sensitive_attr):
        if hasattr(X, "toarray"):
            X = X.toarray()
        X = X.astype(np.float64)
        
        if hasattr(sensitive_attr, 'values'):
            S = sensitive_attr.values
        else:
            S = np.array(sensitive_attr)
        
        n, d = X.shape
        unique_groups = np.unique(S)
        n_candidates = max(int(n * self.candidate_fraction), 100)
        
        # Initialize with K-Means
        kmeans_init = KMeans(n_clusters=self.n_clusters,
                            init='k-means++',
                            n_init=10,
                            random_state=self.random_state)
        labels = kmeans_init.fit_predict(X)
        
        for iteration in range(self.max_iter):
            labels_old = labels.copy()
            
            centers = np.array([X[labels == c].mean(axis=0) 
                               for c in range(self.n_clusters)])
            
            # Precompute cluster statistics
            cluster_stats = self._precompute_cluster_stats(X, labels, S, centers)
            
            # Find candidate points
            candidates = self._find_candidates(X, labels, centers, n_candidates)
            
            n_reassigned = 0
            
            for idx in candidates:
                current_cluster = labels[idx]
                current_point = X[idx]
                current_group = S[idx]
                
                if cluster_stats[current_cluster]['n'] <= 1:
                    continue
                
                best_cluster = current_cluster
                best_score = -np.inf
                
                for c in range(self.n_clusters):
                    if c == current_cluster:
                        continue
                    
                    dist_current = np.linalg.norm(current_point - centers[current_cluster])
                    dist_new = np.linalg.norm(current_point - centers[c])
                    dist_gain = dist_current - dist_new

                    if dist_gain < -1.0:
                        continue
                    

                    fairness_current = cluster_stats[current_cluster]['fairness']
                    fairness_new = cluster_stats[c]['fairness']

                    fairness_gain = (fairness_current - fairness_new) * 0.5
                    score = fairness_gain + 0.1 * dist_gain
                    
                    if score > best_score:
                        best_score = score
                        best_cluster = c
                
                if best_cluster != current_cluster and best_score > 0:
                    labels[idx] = best_cluster
                    n_reassigned += 1
                    
                    cluster_stats[current_cluster]['n'] -= 1
                    cluster_stats[best_cluster]['n'] += 1
            
            n_changed = np.sum(labels != labels_old)
            pct_changed = n_changed / n * 100
            
            if iteration >= self.min_iter:
                if n_changed == 0:
                    break
                
                if pct_changed < self.tol * 100:
                    break
        
        centers = np.array([X[labels == c].mean(axis=0) 
                           for c in range(self.n_clusters)])
        
        inertia = 0.0
        for c in range(self.n_clusters):
            cluster_points = X[labels == c]
            if len(cluster_points) > 0:
                inertia += np.sum((cluster_points - centers[c]) ** 2)
        
        self.labels_ = labels
        self.cluster_centers_ = centers
        self.n_iter_ = iteration + 1
        self.inertia_ = inertia
        
        return self


def fair_centroid_clustering(X, sensitive_attr, k, max_iter=20, min_iter=5):
    model = OptimizedFairCentroid(
        n_clusters=k,
        max_iter=max_iter,
        min_iter=min_iter,
        tol=0.005,
        candidate_fraction=0.05,
        random_state=0
    )
    model.fit(X, sensitive_attr)
    return model.labels_, model.cluster_centers_


# 5. FAIRNESS-AWARE POSTPROCESSING
# 5.1 PP-NFP
class NearForeignFairKMeans:
    def __init__(self, n_clusters=6, balance_tolerance=0.05, max_iter=30, random_state=0):
        self.n_clusters = n_clusters
        self.balance_tolerance = balance_tolerance
        self.max_iter = max_iter
        self.random_state = random_state
        self.labels_ = None
        self.cluster_centers_ = None
        self.inertia_ = None
    
    def fit(self, X, sensitive_attr):
        # Convert to dense
        if hasattr(X, "toarray"):
            X = X.toarray()
        X = X.astype(np.float64)
        
        if hasattr(sensitive_attr, 'values'):
            S = sensitive_attr.values
        else:
            S = np.array(sensitive_attr)
        
        # Standard K-means
        kmeans = KMeans(n_clusters=self.n_clusters, random_state=self.random_state, n_init=10)
        labels = kmeans.fit_predict(X)
        centers = kmeans.cluster_centers_
        
        # Balance adjustment
        for iteration in range(self.max_iter):
            # Compute distances to all centers
            distances = cdist(X, centers, metric='euclidean')
            
            # Find imbalanced clusters
            balances = []
            for c in range(self.n_clusters):
                cluster_mask = (labels == c)
                S_c = S[cluster_mask]
                
                if len(S_c) == 0:
                    balances.append((c, 1.0, 0))
                    continue
                
                n_group0 = np.sum(S_c == 0)
                n_group1 = np.sum(S_c == 1)
                
                if n_group0 > 0 and n_group1 > 0:
                    ratio = n_group0 / n_group1
                    balances.append((c, ratio, len(S_c)))
                else:
                    balances.append((c, float('inf') if n_group0 > 0 else 0, len(S_c)))
            
            # Sort by imbalance
            balances.sort(key=lambda x: abs(np.log(x[1]) if x[1] > 0 and x[1] != float('inf') else 10), reverse=True)
            
            if len(balances) < 2:
                break
            
            # Get the most imbalanced cluster
            cluster_A = balances[0][0]
            ratio_A = balances[0][1]
            
            # Check if balanced enough
            if abs(np.log(ratio_A) if ratio_A > 0 and ratio_A != float('inf') else 0) < self.balance_tolerance:
                break
            
            # Find near-foreign points of cluster A
            mask_A = (labels == cluster_A)
            if mask_A.sum() == 0:
                break
            
            indices_A = np.where(mask_A)[0]
            distances_A = distances[indices_A, cluster_A]
            n_candidates = max(1, int(0.1 * len(indices_A)))
            far_indices = indices_A[np.argsort(distances_A)[-n_candidates:]]
            
            # Try reassigning to nearer clusters
            swapped = False
            for idx in far_indices:
                current_dist = distances[idx, cluster_A]
                sorted_clusters = np.argsort(distances[idx])
                
                for new_cluster in sorted_clusters[1:3]:  # Try 2 nearest other clusters
                    if distances[idx, new_cluster] < current_dist * 1.1:  # Within 10% distance
                        labels[idx] = new_cluster
                        swapped = True
                        break
            
            if not swapped:
                break
            
            # Update centers
            for i in range(self.n_clusters):
                cluster_points = X[labels == i]
                if len(cluster_points) > 0:
                    centers[i] = np.mean(cluster_points, axis=0)
        
        # Compute final inertia
        inertia = 0.0
        for c in range(self.n_clusters):
            cluster_points = X[labels == c]
            if len(cluster_points) > 0:
                inertia += np.sum((cluster_points - centers[c]) ** 2)
        
        self.labels_ = labels
        self.cluster_centers_ = centers
        self.inertia_ = inertia
        
        return self

# PP-Gini
class GiniFairKMeans:
    def __init__(self, n_clusters=6, k_neighbors=10, balance_tolerance=0.05, max_iter=30, random_state=0):
        self.n_clusters = n_clusters
        self.k_neighbors = k_neighbors
        self.balance_tolerance = balance_tolerance
        self.max_iter = max_iter
        self.random_state = random_state
        self.labels_ = None
        self.cluster_centers_ = None
        self.inertia_ = None
    
    def _compute_gini_index(self, X, labels, k):
        n = len(X)
        
        # Find nearest neighbors
        nbrs = NearestNeighbors(n_neighbors=k+1, algorithm='auto').fit(X)
        distances, indices = nbrs.kneighbors(X)
        
        # Compute Gini for each point
        gini_scores = np.zeros(n)
        
        for i in range(n):
            neighbor_labels = labels[indices[i, 1:]]
            unique, counts = np.unique(neighbor_labels, return_counts=True)
            proportions = counts / k
            gini = np.sum(proportions * (1 - proportions))
            gini_scores[i] = gini
        
        return gini_scores

    # Compute Gini coefficient from group counts
    def _compute_gini_coefficient(self, group_counts):
        counts = [c for c in group_counts if c > 0]
        if len(counts) <= 1:
            return 1.0  # Complete inequality
        
        counts = sorted(counts)
        n = sum(counts)
        cumsum = np.cumsum(counts)
        gini = (2 * np.sum((np.arange(1, len(counts) + 1) * counts))) / (n * len(counts)) - (len(counts) + 1) / len(counts)
        return abs(gini)

    # Compute average Gini across all clusters
    def _compute_overall_gini(self, labels, S, k):
        ginis = []
        for c in range(k):
            mask = (labels == c)
            S_c = S[mask]
            if len(S_c) == 0:
                continue
            
            n_group0 = np.sum(S_c == 0)
            n_group1 = np.sum(S_c == 1)
            group_counts = [n_group0, n_group1]
            gini = self._compute_gini_coefficient(group_counts)
            ginis.append(gini)
        
        return np.mean(ginis) if ginis else 0.0
    
    def fit(self, X, sensitive_attr):
        # Convert to dense
        if hasattr(X, "toarray"):
            X = X.toarray()
        X = X.astype(np.float64)
        
        if hasattr(sensitive_attr, 'values'):
            S = sensitive_attr.values
        else:
            S = np.array(sensitive_attr)
        
        # Standard K-means
        kmeans = KMeans(n_clusters=self.n_clusters, random_state=self.random_state, n_init=10)
        labels = kmeans.fit_predict(X)
        centers = kmeans.cluster_centers_
        
        # Gini-based adjustment
        for iteration in range(self.max_iter):
            # Compute distances to all centers
            distances = cdist(X, centers, metric='euclidean')
            
            # Find cluster with highest Gini
            cluster_ginis = []
            for c in range(self.n_clusters):
                cluster_mask = (labels == c)
                S_c = S[cluster_mask]
                
                if len(S_c) == 0:
                    cluster_ginis.append((c, 0.0, 0))
                    continue
                
                n_group0 = np.sum(S_c == 0)
                n_group1 = np.sum(S_c == 1)
                group_counts = [n_group0, n_group1]
                gini = self._compute_gini_coefficient(group_counts)
                cluster_ginis.append((c, gini, len(S_c)))
            
            # Sort by Gini
            cluster_ginis.sort(key=lambda x: x[1], reverse=True)
            
            if len(cluster_ginis) < 2:
                break
            
            # Get cluster with the highest Gini
            cluster_A = cluster_ginis[0][0]
            gini_A = cluster_ginis[0][1]
            
            # Check if Gini < 0.05
            if gini_A < self.balance_tolerance:
                break
            
            # Find near-foreign points of cluster A
            mask_A = (labels == cluster_A)
            if mask_A.sum() == 0:
                break
            
            indices_A = np.where(mask_A)[0]
            distances_A = distances[indices_A, cluster_A]
            n_candidates = max(1, int(0.1 * len(indices_A)))
            far_indices = indices_A[np.argsort(distances_A)[-n_candidates:]]
            
            # Compute current overall Gini
            current_gini = self._compute_overall_gini(labels, S, self.n_clusters)
            
            # Try reassigning to nearer clusters
            swapped = False
            for idx in far_indices:
                current_dist = distances[idx, cluster_A]
                sorted_clusters = np.argsort(distances[idx])
                
                old_label = labels[idx]
                
                for new_cluster in sorted_clusters[1:3]:  # Try 2 nearest other clusters
                    if distances[idx, new_cluster] < current_dist * 1.2:  # Within 20% distance
                        # Try the reassignment
                        labels[idx] = new_cluster
                        new_gini = self._compute_overall_gini(labels, S, self.n_clusters)
                        
                        # Accept if Gini improves
                        if new_gini < current_gini:
                            swapped = True
                            current_gini = new_gini  # Update for next point
                            break
                        else:
                            # Revert
                            labels[idx] = old_label
            
            if not swapped:
                break
            
            # Update centers
            for i in range(self.n_clusters):
                cluster_points = X[labels == i]
                if len(cluster_points) > 0:
                    centers[i] = np.mean(cluster_points, axis=0)
        
        # Compute final inertia
        inertia = 0.0
        for c in range(self.n_clusters):
            cluster_points = X[labels == c]
            if len(cluster_points) > 0:
                inertia += np.sum((cluster_points - centers[c]) ** 2)
        
        self.labels_ = labels
        self.cluster_centers_ = centers
        self.inertia_ = inertia
        
        return self


def postprocessing_clustering_nfp(X, sensitive_attr, k, max_iter=30):
    model = NearForeignFairKMeans(
        n_clusters=k,
        balance_tolerance=0.05,
        max_iter=max_iter,
        random_state=0
    )
    model.fit(X, sensitive_attr)
    return model.labels_, model.cluster_centers_


def postprocessing_clustering_gini(X, sensitive_attr, k, max_iter=30):
    model = GiniFairKMeans(
        n_clusters=k,
        k_neighbors=10,
        balance_tolerance=0.05,
        max_iter=max_iter,
        random_state=0
    )
    model.fit(X, sensitive_attr)
    return model.labels_, model.cluster_centers_


# 6. RAWLSIAN K-MEANS
def rawlsian_kmeans(X, sensitive_attr, k, n_runs=30, delta=None, random_state=0):
    if hasattr(X, "toarray"):
        X_dense = X.toarray()
    else:
        X_dense = X
    
    if hasattr(sensitive_attr, 'values'):
        S_array = sensitive_attr.values
    else:
        S_array = sensitive_attr
    
    # Set delta if not provided
    if delta is None:
        delta = np.sqrt(X_dense.shape[1])
    
    best_rawlsian_score = -np.inf
    best_labels = None
    best_centers = None
    
    for seed in range(n_runs):
        # Run K-means with different seeds
        kmeans = KMeans(n_clusters=k, random_state=random_state+seed, n_init=1)
        labels = kmeans.fit_predict(X_dense)
        centers = kmeans.cluster_centers_
        
        # Compute individual utilities
        distances = np.linalg.norm(X_dense - centers[labels], axis=1)
        utilities = np.maximum(0, 1 - distances / delta)
        
        # Compute group utilities
        unique_groups = np.unique(S_array)
        group_utilities = []
        for g in unique_groups:
            mask_g = (S_array == g)
            if mask_g.sum() > 0:
                group_utilities.append(utilities[mask_g].mean())
        
        # Rawlsian score = minimum group utility (maximin)
        if len(group_utilities) > 0:
            rawlsian_score = min(group_utilities)
        else:
            rawlsian_score = 0
        
        # Track best
        if rawlsian_score > best_rawlsian_score:
            best_rawlsian_score = rawlsian_score
            best_labels = labels.copy()
            best_centers = centers.copy()
    
    return best_labels, best_centers

## 6. Fairness and Quality Metrics

In [16]:
def calculate_quality_metrics(X, labels):
    k = len(np.unique(labels))
    n_samples = len(X)

    
    # Inertia (SSE / WCSS equivalent)
    cluster_centers = []
    inertia = 0
    
    for cluster_id in range(k):
        cluster_mask = (labels == cluster_id)
        cluster_points = X[cluster_mask]
        
        if len(cluster_points) > 0:
            center = np.mean(cluster_points, axis=0)
            cluster_centers.append(center)
            inertia += np.sum((cluster_points - center) ** 2)
        else:
            cluster_centers.append(np.zeros(X.shape[1]))
    
    cluster_centers = np.array(cluster_centers)

    
    # 2. Silhouette Score
    if k > 1 and k < len(X):
        try:
            sil_score = silhouette_score(X, labels)
        except:
            sil_score = 0
    else:
        sil_score = 0
    
    
    # 3. Calinski-Harabasz Index
    if k > 1 and k < len(X):
        try:
            ch_score = calinski_harabasz_score(X, labels)
        except:
            ch_score = 0
    else:
        ch_score = 0
    

    # 4. Davies-Bouldin Index
    if k > 1 and k < len(X):
        try:
            db_score = davies_bouldin_score(X, labels)
        except:
            db_score = 0
    else:
        db_score = 0
    

    # 5. Dunn Index
    dunn_index = 0
    if k > 1:
        try:
            # Minimum inter-cluster distance
            inter_cluster_dists = cdist(cluster_centers, cluster_centers, metric='euclidean')
            np.fill_diagonal(inter_cluster_dists, np.inf)
            min_inter_dist = np.min(inter_cluster_dists)
            
            # Maximum intra-cluster distance (diameter)
            max_intra_dist = 0.0
            for cluster_id in range(k):
                cluster_points = X[labels == cluster_id]
                if len(cluster_points) > 1:
                    pairwise_dists = pdist(cluster_points, metric='euclidean')
                    if len(pairwise_dists) > 0:
                        max_intra_dist = max(max_intra_dist, np.max(pairwise_dists))
            
            if max_intra_dist > 0:
                dunn_index = min_inter_dist / max_intra_dist
        except:
            dunn_index = 0
    

    # 6. WCSS, BCSS, TSS, Variance Ratio
    overall_mean = np.mean(X, axis=0)
    tss = np.sum((X - overall_mean) ** 2)
    
    wcss = 0.0
    bcss = 0.0
    
    for cluster_id in range(k):
        cluster_points = X[labels == cluster_id]
        n_c = len(cluster_points)
        
        if n_c > 0:
            cluster_mean = np.mean(cluster_points, axis=0)
            
            # WCSS
            wcss += np.sum((cluster_points - cluster_mean) ** 2)
            
            # BCSS
            bcss += n_c * np.sum((cluster_mean - overall_mean) ** 2)
    
    variance_ratio = bcss / wcss if wcss > 0 else 0.0
    

    # 7. Inter-Cluster Distance
    inter_cluster_distance = 0
    if k > 1:
        try:
            pairwise_dists = pdist(cluster_centers, metric='euclidean')
            inter_cluster_distance = np.mean(pairwise_dists) if len(pairwise_dists) > 0 else 0
        except:
            inter_cluster_distance = 0
    
  
    # 8. Intra-Cluster Distance
    intra_distances = []
    for cluster_id in range(k):
        cluster_points = X[labels == cluster_id]
        if len(cluster_points) > 1:
            try:
                pairwise_dists = pdist(cluster_points, metric='euclidean')
                intra_distances.append(np.mean(pairwise_dists))
            except:
                pass
    
    intra_cluster_distance = np.mean(intra_distances) if intra_distances else 0
    

    # 9. Separation Index
    separation_index = 0
    if intra_cluster_distance > 0:
        separation_index = inter_cluster_distance / intra_cluster_distance
    

    # Return all
    return {
        'Inertia': inertia,
        'Silhouette': sil_score,
        'Calinski_Harabasz': ch_score,
        'Davies_Bouldin': db_score,
        'Dunn_Index': dunn_index,
        'WCSS': wcss,
        'BCSS': bcss,
        'TSS': tss,
        'Variance_Ratio': variance_ratio,
        'Inter_Cluster_Distance': inter_cluster_distance,
        'Intra_Cluster_Distance': intra_cluster_distance,
        'Separation_Index': separation_index
    }


def calculate_external_metrics(labels_true, labels_pred):
    try:
        ari = adjusted_rand_score(labels_true, labels_pred)
    except:
        ari = 0.0
    
    try:
        nmi = normalized_mutual_info_score(labels_true, labels_pred)
    except:
        nmi = 0.0
    
    try:
        ami = adjusted_mutual_info_score(labels_true, labels_pred)
    except:
        ami = 0.0
    
    try:
        fmi = fowlkes_mallows_score(labels_true, labels_pred)
    except:
        fmi = 0.0
    
    try:
        v_measure = v_measure_score(labels_true, labels_pred)
    except:
        v_measure = 0.0
    
    return {
        'ARI': ari,
        'NMI': nmi,
        'AMI': ami,
        'FMI': fmi,
        'V-Measure': v_measure
    }


# FAIRNESS METRICS
# Balance
def calculate_balance(labels, sensitive_attr):
    k = len(np.unique(labels))
    balances = []
    
    for cluster_id in range(k):
        cluster_mask = (labels == cluster_id)
        cluster_sensitive = sensitive_attr[cluster_mask]
        
        if len(cluster_sensitive) == 0:
            continue
        
        n_group0 = np.sum(cluster_sensitive == 0)
        n_group1 = np.sum(cluster_sensitive == 1)
        
        if n_group0 > 0 and n_group1 > 0:
            balance = min(n_group0, n_group1) / max(n_group0, n_group1)
            balances.append(balance)
    
    return np.mean(balances) if balances else 0.0


# Statistical Parity Difference
def calculate_spd(labels, sensitive_attr):
    k = len(np.unique(labels))
    total_group0 = np.sum(sensitive_attr == 0)
    total_group1 = np.sum(sensitive_attr == 1)
    
    spd_values = []
    
    for cluster_id in range(k):
        cluster_mask = (labels == cluster_id)
        n_group0 = np.sum((sensitive_attr == 0) & cluster_mask)
        n_group1 = np.sum((sensitive_attr == 1) & cluster_mask)
        
        p0 = n_group0 / total_group0 if total_group0 > 0 else 0
        p1 = n_group1 / total_group1 if total_group1 > 0 else 0
        
        spd = abs(p0 - p1)
        spd_values.append(spd)
    
    return np.mean(spd_values)


# Disparate Impact
def calculate_disparate_impact(labels, sensitive_attr):
    k = len(np.unique(labels))
    total_group0 = np.sum(sensitive_attr == 0)
    total_group1 = np.sum(sensitive_attr == 1)
    
    di_values = []
    
    for cluster_id in range(k):
        cluster_mask = (labels == cluster_id)
        n_group0 = np.sum((sensitive_attr == 0) & cluster_mask)
        n_group1 = np.sum((sensitive_attr == 1) & cluster_mask)
        
        p0 = n_group0 / total_group0 if total_group0 > 0 else 0
        p1 = n_group1 / total_group1 if total_group1 > 0 else 0
        
        if p0 > 0 and p1 > 0:
            di = min(p0, p1) / max(p0, p1)
            di_values.append(di)
    
    return np.mean(di_values) if di_values else 0.0


# Entropy
def calculate_entropy(labels, sensitive_attr):
    k = len(np.unique(labels))
    entropies = []
    
    for cluster_id in range(k):
        cluster_mask = (labels == cluster_id)
        cluster_sensitive = sensitive_attr[cluster_mask]
        
        if len(cluster_sensitive) == 0:
            continue
        
        n_group0 = np.sum(cluster_sensitive == 0)
        n_group1 = np.sum(cluster_sensitive == 1)
        
        if n_group0 > 0 and n_group1 > 0:
            probs = np.array([n_group0, n_group1]) / len(cluster_sensitive)
            ent = entropy(probs, base=2)
            entropies.append(ent)
    
    # Normalize by max entropy
    return np.mean(entropies) if entropies else 0.0


# EVALUATION FUNCTION
def evaluate_clustering(X, labels, sensitive_attr=None, labels_true=None):
    results = {}
    
    # Quality metrics (Internal - 12个)
    results['quality'] = calculate_quality_metrics(X, labels)
    
    # External metrics (5个)
    if labels_true is not None:
        results['external'] = calculate_external_metrics(labels_true, labels)
    
    # Fairness metrics (4个)
    if sensitive_attr is not None:
        results['fairness'] = {
            'Balance': calculate_balance(labels, sensitive_attr),
            'SPD': calculate_spd(labels, sensitive_attr),
            'Disparate_Impact': calculate_disparate_impact(labels, sensitive_attr),
            'Entropy': calculate_entropy(labels, sensitive_attr)
        }
    
    return results


# EXPERIMENT FRAMEWORK
def run_clustering_experiment(dataset_name, X, sensitive_attr, k, method_name, 
                              labels_true=None, method_function=None):
    # Run clustering
    if method_function is None:
        raise ValueError("method_function must be provided")
    
    labels, centers = method_function(X, sensitive_attr, k)
    
    # Calculate all metrics
    metrics = evaluate_clustering(X, labels, sensitive_attr, labels_true)
    
    return {
        'dataset': dataset_name,
        'method': method_name,
        'k': k,
        'labels': labels,
        'centers': centers,
        'quality': metrics['quality'],
        'external': metrics.get('external', {}),
        'fairness': metrics.get('fairness', {})
    }


def create_comparison_tables(results, approach_name):
    dataset_names = list(results.keys())
    methods = list(next(iter(results.values())).keys())

    # Quality Metrics Table
    quality_data = []
    for dataset_name in dataset_names:
        for method in methods:
            if method in results[dataset_name]:
                res = results[dataset_name][method]
                quality_data.append({
                    'Dataset': dataset_name.upper(),
                    'Method': method.upper(),
                    'k': res['k'],
                    'Inertia': res['quality']['Inertia'],
                    'Silhouette': res['quality']['Silhouette'],
                    'Calinski_Harabasz': res['quality']['Calinski_Harabasz'],
                    'Davies_Bouldin': res['quality']['Davies_Bouldin'],
                    'Dunn_Index': res['quality']['Dunn_Index'],
                    'WCSS': res['quality']['WCSS'],
                    'BCSS': res['quality']['BCSS'],
                    'TSS': res['quality']['TSS'],
                    'Variance_Ratio': res['quality']['Variance_Ratio'],
                    'Inter_Cluster_Distance': res['quality']['Inter_Cluster_Distance'],
                    'Intra_Cluster_Distance': res['quality']['Intra_Cluster_Distance'],
                    'Separation_Index': res['quality']['Separation_Index']
                })
    
    quality_df = pd.DataFrame(quality_data)
    
    # External Metrics Table
    for dataset_name in dataset_names:
        for method in methods:
            if method in results[dataset_name] and results[dataset_name][method]['external']:
                res = results[dataset_name][method]
                external_data.append({
                    'Dataset': dataset_name.upper(),
                    'Method': method.upper(),
                    'k': res['k'],
                    'ARI': res['external']['ARI'],
                    'NMI': res['external']['NMI'],
                    'AMI': res['external']['AMI'],
                    'FMI': res['external']['FMI'],
                    'V-Measure': res['external']['V-Measure']
                })
    
    external_df = pd.DataFrame(external_data) if external_data else None
    
    # Fairness Metrics Table
    fairness_data = []
    for dataset_name in dataset_names:
        for method in methods:
            if method in results[dataset_name]:
                res = results[dataset_name][method]
                fairness_data.append({
                    'Dataset': dataset_name.upper(),
                    'Method': method.upper(),
                    'k': res['k'],
                    'Balance': res['fairness']['Balance'],
                    'SPD': res['fairness']['SPD'],
                    'Disparate_Impact': res['fairness']['Disparate_Impact'],
                    'Entropy': res['fairness']['Entropy']
                })
    
    fairness_df = pd.DataFrame(fairness_data)
    
    return quality_df, external_df, fairness_df

## 7. Run Experiments - Apply Optimal K to All Methods

In [19]:
def run_clustering_experiment(dataset_name, X, sensitive_attr, k, method_name):
    if method_name == 'kmeans':
        labels, centers = standard_kmeans(X, k)
    elif method_name == 'fairlet':
        labels, centers = fairlet_clustering(X, sensitive_attr, k, p=1, q=1)
    elif method_name == 'bfkm':
        labels, centers = bfkm_clustering(X, sensitive_attr, k)
    elif method_name == 'fair_centroid':
        labels, centers = fair_centroid_clustering(X, sensitive_attr, k, max_iter=10)
    elif method_name == 'postprocessing_nfp':
        labels, centers = postprocessing_clustering_nfp(X, sensitive_attr, k, max_iter=15)
    elif method_name == 'postprocessing_gini':
        labels, centers = postprocessing_clustering_gini(X, sensitive_attr, k, max_iter=15)
    elif method_name == 'rawlsian':
        labels, centers = rawlsian_kmeans(X, sensitive_attr, k, n_runs=20)
    else:
        raise ValueError(f"Unknown method: {method_name}")
    
    # Calculate metrics
    quality = calculate_quality_metrics(X, labels)
    fairness = {
        'Balance': calculate_balance(labels, sensitive_attr),
        'SPD': calculate_spd(labels, sensitive_attr),
        'Disparate_Impact': calculate_disparate_impact(labels, sensitive_attr),
        'Entropy': calculate_entropy(labels, sensitive_attr)
    }
    return {
        'dataset': dataset_name,
        'method': method_name,
        'k': k,
        'labels': labels,
        'centers': centers,
        'quality': quality,
        'fairness': fairness
    }


# Main experiment loop
all_results = {
    'approach1': {},
    'approach2': {}
}

methods = ['kmeans', 'fairlet', 'bfkm','fair_centroid', 'postprocessing_nfp','postprocessing_gini', 'rawlsian']

print("\n" + "="*80)
print("RUNNING CLUSTERING EXPERIMENTS")
print("="*80)

for dataset_name in dataset_names:
    X = datasets[dataset_name]['X']
    sensitive = datasets[dataset_name]['sensitive']
    
    print(f"\n{'='*80}")
    print(f"Dataset: {dataset_name.upper()}")
    print(f"{'='*80}")
    
    # Approach 1 optimal k
    k_approach1 = approach1_results[dataset_name]['optimal_k']
    print(f"\nApproach 1 - Optimal k = {k_approach1}")
    all_results['approach1'][dataset_name] = {}
    
    for method in methods:
        print(f"  Running {method.upper()}...")
        result = run_clustering_experiment(dataset_name, X, sensitive, k_approach1, method)
        all_results['approach1'][dataset_name][method] = result
        
        print(f"    Quality - Silhouette: {result['quality']['Silhouette']:.4f}")
        print(f"    Fairness - Balance: {result['fairness']['Balance']:.4f}")
    
    # Approach 2 optimal k
    k_approach2 = approach2_results[dataset_name]['majority_vote_k']
    print(f"\nApproach 2 - Optimal k = {k_approach2}")
    all_results['approach2'][dataset_name] = {}
    
    for method in methods:
        print(f"  Running {method.upper()}...")
        result = run_clustering_experiment(dataset_name, X, sensitive, k_approach2, method)
        all_results['approach2'][dataset_name][method] = result
        
        print(f"    Quality - Silhouette: {result['quality']['Silhouette']:.4f}")
        print(f"    Fairness - Balance: {result['fairness']['Balance']:.4f}")

print("\n" + "="*80)
print("ALL EXPERIMENTS COMPLETED!")
print("="*80)


RUNNING CLUSTERING EXPERIMENTS

Dataset: ADULT

Approach 1 - Optimal k = 5
  Running KMEANS...
    Quality - Silhouette: 0.1584
    Fairness - Balance: 0.3723
  Running FAIRLET...
    Quality - Silhouette: -0.0063
    Fairness - Balance: 0.8435
  Running BFKM...
    Quality - Silhouette: 0.1332
    Fairness - Balance: 0.3869
  Running FAIR_CENTROID...
    Quality - Silhouette: 0.1618
    Fairness - Balance: 0.4402
  Running POSTPROCESSING_NFP...
    Quality - Silhouette: 0.1625
    Fairness - Balance: 0.4398
  Running POSTPROCESSING_GINI...
    Quality - Silhouette: 0.1516
    Fairness - Balance: 0.5054
  Running RAWLSIAN...
    Quality - Silhouette: 0.1306
    Fairness - Balance: 0.3854

Approach 2 - Optimal k = 5
  Running KMEANS...
    Quality - Silhouette: 0.1584
    Fairness - Balance: 0.3723
  Running FAIRLET...
    Quality - Silhouette: -0.0130
    Fairness - Balance: 0.8413
  Running BFKM...
    Quality - Silhouette: 0.1144
    Fairness - Balance: 0.3989
  Running FAIR_CENTROI

## 8. Save Results to PKL Files

In [21]:
# Save all results
with open('optimal_k_results.pkl', 'wb') as f:
    pickle.dump({
        'approach1_k': approach1_results,
        'approach2_k': approach2_results,
        'clustering_results': all_results,
        'datasets': dataset_names
    }, f)

print("\n✓ Results saved to: optimal_k_results.pkl")

# Save individual approach results
with open('approach1_clustering_results.pkl', 'wb') as f:
    pickle.dump(all_results['approach1'], f)
    
with open('approach2_clustering_results.pkl', 'wb') as f:
    pickle.dump(all_results['approach2'], f)

print("✓ Individual results saved to:")
print("  - approach1_clustering_results.pkl")
print("  - approach2_clustering_results.pkl")


✓ Results saved to: optimal_k_results.pkl
✓ Individual results saved to:
  - approach1_clustering_results.pkl
  - approach2_clustering_results.pkl


## 9. Create Comparison Tables

In [32]:
def create_comparison_tables(results, approach_name):
    dataset_names = list(results.keys())
    methods = list(next(iter(results.values())).keys())
    

    # Quality Metrics Table
    quality_data = []
    for dataset_name in dataset_names:
        for method in methods:
            if method in results[dataset_name]:
                res = results[dataset_name][method]
                quality_data.append({
                    'Dataset': dataset_name.upper(),
                    'Method': method.upper(),
                    'k': res['k'],
                    'Inertia': res['quality']['Inertia'],
                    'Silhouette': res['quality']['Silhouette'],
                    'Calinski_Harabasz': res['quality']['Calinski_Harabasz'],
                    'Davies_Bouldin': res['quality']['Davies_Bouldin'],
                    'Dunn_Index': res['quality']['Dunn_Index'],
                    'WCSS': res['quality']['WCSS'],
                    'BCSS': res['quality']['BCSS'],
                    'TSS': res['quality']['TSS'],
                    'Variance_Ratio': res['quality']['Variance_Ratio'],
                    'Inter_Cluster_Distance': res['quality']['Inter_Cluster_Distance'],
                    'Intra_Cluster_Distance': res['quality']['Intra_Cluster_Distance'],
                    'Separation_Index': res['quality']['Separation_Index']
                })
    
    quality_df = pd.DataFrame(quality_data)
    

    # External Metrics Table
    external_data = []
    for dataset_name in dataset_names:
        for method in methods:
            if method in results[dataset_name]:
                res = results[dataset_name][method]
                if res.get('external') and len(res['external']) > 0:
                    external_data.append({
                        'Dataset': dataset_name.upper(),
                        'Method': method.upper(),
                        'k': res['k'],
                        'ARI': res['external']['ARI'],
                        'NMI': res['external']['NMI'],
                        'AMI': res['external']['AMI'],
                        'FMI': res['external']['FMI'],
                        'V-Measure': res['external']['V-Measure']
                    })

    external_df = pd.DataFrame(external_data) if external_data else None
    

    # Fairness Metrics Table
    fairness_data = []
    for dataset_name in dataset_names:
        for method in methods:
            if method in results[dataset_name]:
                res = results[dataset_name][method]
                fairness_data.append({
                    'Dataset': dataset_name.upper(),
                    'Method': method.upper(),
                    'k': res['k'],
                    'Balance': res['fairness']['Balance'],
                    'SPD': res['fairness']['SPD'],
                    'Disparate_Impact': res['fairness']['Disparate_Impact'],
                    'Entropy': res['fairness']['Entropy']
                })
    
    fairness_df = pd.DataFrame(fairness_data)
    
    return quality_df, external_df, fairness_df


def create_and_save_tables(all_results, approach_name):
    print(f"\n{'='*100}")
    print(f"{approach_name.upper()} RESULTS")
    print(f"{'='*100}")
    
    # Create tables
    quality_df, external_df, fairness_df = create_comparison_tables(
        all_results[approach_name], 
        approach_name
    )
    
    print("\nQuality Metrics (12 metrics):")
    print(quality_df.to_string(index=False))
    
    if external_df is not None and len(external_df) > 0:
        print("\nExternal Metrics (5 metrics):")
        print(external_df.to_string(index=False))
    else:
        print("\nExternal Metrics: Not available (no ground truth labels)")
    
    print("\nFairness Metrics (4 metrics):")
    print(fairness_df.to_string(index=False))
    
    # Save CSV
    quality_df.to_csv(f'{approach_name}_quality_metrics.csv', index=False)
    
    if external_df is not None and len(external_df) > 0:
        external_df.to_csv(f'{approach_name}_external_metrics.csv', index=False)
    
    fairness_df.to_csv(f'{approach_name}_fairness_metrics.csv', index=False)
    
    print(f"\n✓ Tables saved for {approach_name}!")
    
    return quality_df, external_df, fairness_df

quality_df1, external_df1, fairness_df1 = create_and_save_tables(all_results, 'approach1')
quality_df2, external_df2, fairness_df2 = create_and_save_tables(all_results, 'approach2')


APPROACH1 RESULTS

Quality Metrics (12 metrics):
Dataset              Method  k       Inertia  Silhouette  Calinski_Harabasz  Davies_Bouldin  Dunn_Index          WCSS          BCSS      TSS  Variance_Ratio  Inter_Cluster_Distance  Intra_Cluster_Distance  Separation_Index
  ADULT              KMEANS  5 243602.073317    0.158367        3662.568871        1.592339           0 243602.073317 118341.926683 361944.0        0.485800                       0                       0                 0
  ADULT             FAIRLET  5 330496.479298   -0.006316         717.377447        3.790500           0 330496.479298  31447.520702 361944.0        0.095152                       0                       0                 0
  ADULT                BFKM  5 266884.923734    0.133165        2685.330182        2.008327           0 266884.923734  95059.076266 361944.0        0.356180                       0                       0                 0
  ADULT       FAIR_CENTROID  5 249926.835086    0.161765  